In [ ]:
import polars as pl
from pathlib import Path
import matplotlib.pyplot as plt
from datetime import datetime as dt

from piepy.core.data_functions import make_subsets
from piepy.psychophysics.wheel.detection.wheelDetectionExperimentHub import WheelDetectionExperimentHub
from piepy.plotters.psychophysics.detection.delta_effect_plot import plot_delta_effect_contrast, plot_delta_effect_areas

In [ ]:
DATA_PATH = Path.cwd().parents[0] / "260410_Ncomm_inhibition_data.parquet"

In [ ]:
hub = WheelDetectionExperimentHub()
# load from session list
# hub.set_data(all_sessions,
#              load_sessions=True,
#              make_summary=True)
# hub.data.write_parquet("250206_experiment_data.parquet")

# load from saved data
all_data = pl.read_parquet(DATA_PATH)
hub.set_data(all_data,
             load_sessions=True,
             make_summary=True)

In [ ]:
df = hub.filter_by_animals(['KC139','KC142','KC143',
                            'KC144','KC145','KC146',
                            'KC147','KC148','KC149',
                            'KC150','KC151','KC152'],
                      stim_combination="0.04cpd_8.0Hz",
                      isCNO=False)

valid_sesh_ids = []
for filt_tup in make_subsets(df,["session_id"]):
    filt_df = filt_tup[-1]
    if 1.0 not in filt_df["contrast"].to_list() and 0.0625 not in filt_df["contrast"].to_list():
        print(filt_tup[0],filt_df["contrast"].unique().to_list())
        print(filt_df.filter(pl.col("session_id")==filt_tup[0])[0,"session_path"])
        valid_sesh_ids.append(filt_tup[0]) 
    else:
        pass
    
df.filter(pl.col("session_id").is_in(valid_sesh_ids))

In [ ]:
cm = 1/2.54
contrast = 0.5
effect_on = "reaction"
f,ax = plot_delta_effect_areas(data=df,
                               effect_on=effect_on,
                               effect_metric="delta",
                               plot_with="sem",
                               contrast=contrast,
                               areas = ["LM","AL","RL","PM","AM"],
                               polarity=1,
                               trial_count_identifier="dot_color",
                               trial_bounds=(1,5,10,25,50,100),
                               dot_cmap="Greens",
                               mpl_kwargs={"figsize":(12*cm,12*cm),
                                           "linewidths":0,
                                           "s":80},
                               style="print")

# ax.set_yscale("symlog",linthresh=400)
# # ax.set_yscale('asinh',linear_width=200, base=10)
# major_ticks = [-1000,-100,0,100,1000]
# ax.set_yticks(major_ticks)

# # minor_locator = plt.LogLocator(base=10.0, numticks=99, subs=(0.1,0.2,0.4,0.6,0.8))
# # ax.yaxis.set_minor_locator(minor_locator)

# # negative_minor_ticks = [-t for t in minor_locator.tick_values(1, 500) if t > 0]  # Mirror positive minor ticks
# # all_minor_ticks = sorted(negative_minor_ticks + minor_locator.tick_values(1, 500).tolist())  # Combine
# ax.set_yticks(np.arange(-800,1000,200), minor=True)  # Apply minor ticks symmetrically

# ax.yaxis.set_minor_formatter(plt.FormatStrFormatter("%d"))
# ax.yaxis.set_major_formatter(ticker.FormatStrFormatter("%d"))

ax.set_yticks([-200,0,200,400])
ax.set_ylim([-250,400])

    # ax.set_yticks([0,50,100,150,200])
    # ax.set_ylim([-50,200])

In [ ]:
savename = f"{dt.strftime(dt.today(),'%y%m%d')}_{contrast}_ALLAREAS_delta_reaction_time.pdf"
f.savefig(f"pdf/{savename}.pdf")
f.savefig(f"svg/{savename}.svg")

In [ ]:
areas = ["LM", "AL", "RL", "PM", "AM"]
stim_combination = "0.04cpd_8.0Hz"

data_dict = {}
for a in areas:
    df = hub.filter_by_areas(a,
                        strict_performance=True,
                        stim_combination=stim_combination,
                        isCNO=False)
    data_dict[a] = df

data_dict

In [ ]:
cm = 1/2.54
effect_on = "hit_rate"

fig,axs = plt.subplots(1,len(data_dict),
                       figsize=(12*len(data_dict)*cm,12*cm))

for i,k in enumerate(data_dict.keys()):
    ax = axs[i]
    df = data_dict[k]
    _,ax = plot_delta_effect_contrast(df,
                                      ax = ax,
                                      effect_on=effect_on,
                                      plot_with="sem",
                                      polarity=-1,
                                      cloud_width=0.02,
                                      cloud_offset=0,
                                      include_misses=False,
                                      mpl_kwargs={"figsize":(12*cm,12*cm)},
                                      style="print")
    ax.set_title(k)

plt.tight_layout()

In [ ]:
savename = f"ALLAREAS_{stim_combination}_DELTA_hit_rate.pdf"
fig.savefig(f"pdf/{savename}.pdf")
fig.savefig(f"svg/{savename}.svg")